# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [1]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [2]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/employee"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 3

--- Employee_20260207170557.parquet ---
  Registros: 2
  Colunas: ['raw_value']
  ❌ Não contém dados JSON

--- Employee_20260325043100.parquet ---
  Registros: 10
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']

--- placeholder_employee_20260324.parquet ---
  Registros: 1
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']



import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [3]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema)).select("parsed.*")
        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 2

Comparando: Employee_20260325043100.parquet vs placeholder_employee_20260324.parquet
  ✅ Mesmas colunas (25 colunas)
  ✅ Tipos de dados idênticos

✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!


In [4]:
from pyspark.sql.functions import from_json, schema_of_json, col
from pyspark.sql.types import StructType, StringType

def flatten_df(df, sep="_"):
    """Achata recursivamente colunas StructType em colunas simples."""
    colunas_complexas = True
    while colunas_complexas:
        colunas_complexas = False
        novas_colunas = []
        for field in df.schema.fields:
            if isinstance(field.dataType, StructType):
                colunas_complexas = True
                for sub in field.dataType.fields:
                    novo_nome = f"{field.name}{sep}{sub.name}"
                    novas_colunas.append(df[field.name][sub.name].alias(novo_nome))
            else:
                novas_colunas.append(df[field.name])
        df = df.select(novas_colunas)
    return df

def parse_nested_json(df, sep="_", max_depth=3):
    """Detecta e parseia colunas string que contêm JSON, depois achata."""
    for _ in range(max_depth):
        str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        if not str_cols:
            break

        amostra = df.select(*[df[c] for c in str_cols]).filter(
            df[str_cols[0]].isNotNull()
        ).first()
        if not amostra:
            break

        cols_json = []
        for i, c in enumerate(str_cols):
            val = amostra[i]
            if val and isinstance(val, str) and val.strip() and val.strip()[0] in ('{', '['):
                cols_json.append((c, val))

        if not cols_json:
            break

        for c, val in cols_json:
            schema = schema_of_json(val)
            df = df.withColumn(c, from_json(df[c], schema))

        df = flatten_df(df, sep)
    return df

# Para cada arquivo com JSON, parsear, expandir sub-JSONs e achatar
for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"--- {nome}: Sem JSON, mostrando dados diretos ---")
        info["df"].show(5, truncate=False)
        print()
        continue

    df = info["df"]
    for col_json in info["colunas_json"]:
        print(f"--- {nome} | Coluna: {col_json} ---")
        amostra = df.select(col_json).filter(df[col_json].isNotNull()).first()[0]
        json_schema = schema_of_json(amostra)

        df_parsed = df.withColumn("parsed", from_json(df[col_json], json_schema)) \
                      .select("parsed.*")

        colunas_antes = set(df_parsed.columns)

        # Parsear sub-JSONs e achatar tudo
        df_flat = parse_nested_json(df_parsed)

        colunas_expandidas = [c for c in df_flat.columns if c not in colunas_antes]
        if colunas_expandidas:
            print(f"Colunas JSON aninhadas expandidas: {colunas_expandidas}")

        print(f"Total de colunas: {len(df_flat.columns)}")
        df_flat.printSchema()
        df_flat.show(5, truncate=False)

        info["df_parsed"] = df_flat
    print()

--- Employee_20260207170557.parquet: Sem JSON, mostrando dados diretos ---
+-------------+
|raw_value    |
+-------------+
|QueryResponse|
|time         |
+-------------+


--- Employee_20260325043100.parquet | Coluna: raw_value ---
Colunas JSON aninhadas expandidas: ['MetaData_CreateTime', 'MetaData_LastUpdatedTime', 'MetaData_source', 'Mobile_FreeFormNumber', 'PrimaryAddr_City', 'PrimaryAddr_CountrySubDivisionCode', 'PrimaryAddr_Line1', 'PrimaryAddr_PostalCode', 'PrimaryEmailAddr_Address', 'PrimaryPhone_FreeFormNumber']
Total de colunas: 30
root
 |-- Active: string (nullable = true)
 |-- BillRate: string (nullable = true)
 |-- BillableTime: string (nullable = true)
 |-- BirthDate: string (nullable = true)
 |-- DisplayName: string (nullable = true)
 |-- EmployeeNumber: string (nullable = true)
 |-- FamilyName: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- GivenName: string (nullable = true)
 |-- HiredDate: string (nullable = true)
 |-- Id: string (nullable = true

In [5]:
from IPython.display import display
from functools import reduce
from pyspark.sql.functions import col, count, when

# Juntar todos os DataFrames parseados em um só, tratando diferenças de schema
dfs_parsed = [info["df_parsed"] for info in dataframes.values() if "df_parsed" in info]

if dfs_parsed:
    df_merged = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_parsed)

    # Identificar e remover colunas que são totalmente nulas
    total = df_merged.count()
    nulls_por_coluna = df_merged.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in df_merged.columns]
    ).first()

    colunas_nulas = [c for c in df_merged.columns if nulls_por_coluna[c] == total]
    if colunas_nulas:
        print(f"Colunas totalmente nulas removidas: {colunas_nulas}")
        df_merged = df_merged.drop(*colunas_nulas)

    print(f"Total de linhas: {total} | Total de colunas: {len(df_merged.columns)}")
    display(df_merged.limit(10).toPandas())
else:
    print("Nenhum DataFrame parseado encontrado.")

Colunas totalmente nulas removidas: ['ReleasedDate', 'Suffix', 'MetaData', 'Mobile', 'PrimaryAddr', 'PrimaryEmailAddr', 'PrimaryPhone']
Total de linhas: 11 | Total de colunas: 28


,Active,BillRate,BillableTime,BirthDate,DisplayName,EmployeeNumber,FamilyName,Gender,GivenName,HiredDate,...,PrimaryAddr_Line1,PrimaryAddr_PostalCode,PrimaryEmailAddr_Address,PrimaryPhone_FreeFormNumber,PrintOnCheckName,SSN,SyncToken,Title,domain,sparse
0,true,23.50,true,1990-02-15,Employee 1,E001,Worker1,Male,Emp1,2024-02-01,...,201 Oak Ave,35161,employee1@example.com,555-41001,Employee 1,000-00-1001,401,Dental Assistant,QBO,false
1,true,25.00,false,1990-03-15,Employee 2,E002,Worker2,Female,Emp2,2024-03-01,...,202 Oak Ave,35162,employee2@example.com,555-41002,Employee 2,000-00-1002,402,Hygienist,QBO,false
2,true,26.50,true,1990-04-15,Employee 3,E003,Worker3,Male,Emp3,2024-04-01,...,203 Oak Ave,35163,employee3@example.com,555-41003,Employee 3,000-00-1003,403,Billing Specialist,QBO,false
3,true,28.00,false,1990-05-15,Employee 4,E004,Worker4,Female,Emp4,2024-05-01,...,204 Oak Ave,35164,employee4@example.com,555-41004,Employee 4,000-00-1004,404,Office Manager,QBO,false
4,true,29.50,true,1990-06-15,Employee 5,E005,Worker5,Male,Emp5,2024-06-01,...,205 Oak Ave,35165,employee5@example.com,555-41005,Employee 5,000-00-1005,405,Dental Assistant,QBO,false
5,true,31.00,false,1990-07-15,Employee 6,E006,Worker6,Female,Emp6,2024-07-01,...,206 Oak Ave,35166,employee6@example.com,555-41006,Employee 6,000-00-1006,406,Hygienist,QBO,false
6,true,32.50,true,1990-08-15,Employee 7,E007,Worker7,Male,Emp7,2024-08-01,...,207 Oak Ave,35167,employee7@example.com,555-41007,Employee 7,000-00-1007,407,Billing Specialist,QBO,false
7,true,34.00,false,1990-09-15,Employee 8,E008,Worker8,Female,Emp8,2024-09-01,...,208 Oak Ave,35168,employee8@example.com,555-41008,Employee 8,000-00-1008,408,Office Manager,QBO,false
8,true,35.50,true,1990-10-15,Employee 9,E009,Worker9,Male,Emp9,2024-10-01,...,209 Oak Ave,35169,employee9@example.com,555-41009,Employee 9,000-00-1009,409,Dental Assistant,QBO,false
9,true,37.00,false,1990-11-15,Employee 10,E010,Worker10,Female,Emp10,2024-11-01,...,210 Oak Ave,351610,employee10@example.com,555-41010,Employee 10,000-00-1010,410,Hygienist,QBO,false


In [6]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
